**Code-only release candidate.** Market inputs and executed outputs are deliberately absent pending redistribution clearance. See `DATA_SCHEMA.md` before running market-dependent cells. `python src/teaching_interfaces.py` runs deterministic synthetic checks without market data; it does not reproduce the empirical results.

# Environment and Calendar-Aware Market Data

## Goal

Companion to Chapter 1. Establish what the data and execution environment represent before building a circuit. The fixed eight-stock cohort is AAPL, MSFT, AMZN, GOOGL, META, JPM, GS and XOM, spanning five broad sectors. It is not random, market-representative or a historical constituent reconstruction. Eight decision bits make all 256 basis strings inspectable; selecting four gives 70 feasible baskets. SPX is a descriptive price-index reference.

## Setup

Run this notebook from top to bottom. The experiment uses local simulation only; no cloud account, trading connection or API key is needed. Market observations stop on **31 August 2026**. The supplied data are quoted-price proxies, not verified total-return series.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Resolve only the package itself; no author-specific absolute path is needed.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "daily_market.csv").exists(), "Authorised daily inputs are required; see DATA_SCHEMA.md. Synthetic checks run without market data via src/teaching_interfaces.py."
sys.path.insert(0, str(ROOT / "src"))
from common import DATA, RESULTS, market


## 1. Inspect the data contract

The manifest records Bloomberg as the author-identified provider, distinguishes requested dates from observed sessions, identifies missing files and states unresolved adjustment and redistribution questions. File hashes check identity, not economic authenticity.

In [ ]:
manifest = json.loads((DATA / "manifest.json").read_text())
display(pd.Series({k: manifest[k] for k in ["source", "observed_start", "observed_end", "rows", "expected_sessions", "missing_files_or_sessions", "price_adjustment", "redistribution"]}, name="Data contract").to_frame())

## 2. Check chronology and coverage

Times were converted from the source archive's Shanghai convention to New York, including daylight saving. The window is 09:35--15:55, shortened to 12:55 on exchange-designated early closes. Missing observations are not filled.

In [ ]:
daily = market()
assert daily.date.max() <= pd.Timestamp("2026-08-31")
assert not daily.duplicated(["date", "ticker"]).any()
display(daily.groupby("ticker").agg(sessions=("date", "size"), first=("date", "min"), last=("date", "max")))
display(pd.read_csv(DATA / "data_issues.csv"))

## 3. Verify a minimal circuit

PennyLane and Qiskit represent the same qubit mathematics. Here a Bell state checks preparation and probabilities. Exact simulator amplitudes are not an output that a physical QPU supplies directly.

In [ ]:
import pennylane as qml
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
dev = qml.device("default.qubit", wires=2)
@qml.qnode(dev)
def bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
pl = np.asarray(bell())
qk = Statevector.from_instruction(qc).probabilities()
np.testing.assert_allclose(pl, qk, atol=1e-12)
display(pd.DataFrame({"basis": ["00", "01", "10", "11"], "PennyLane": pl, "Qiskit": qk}))

## 4. Allocate a local register and verify bit order

Eight simulated wires store 256 complex amplitudes locally; Hadamards give 256 equal probabilities. An asymmetric two-qubit X test then makes the SDK ordering difference visible, unlike the reversal-symmetric Bell state. No hardware account is contacted.

In [ ]:
from teaching_interfaces import local_register_probabilities, bit_order_example
p = local_register_probabilities()
print("Eight-wire probability vector:", p.shape, "probability sum:", p.sum())
display(pd.DataFrame(bit_order_example()))

## 5. Record software versions

Use the pinned environment when reproducing numerical outputs. Later SDK releases can change interfaces and compilation, even when the financial equations are unchanged.

In [ ]:
import importlib.metadata as metadata
versions = {p: metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "pennylane", "qiskit"]}
display(pd.Series(versions, name="Installed version").to_frame())

## Interpretation and Next Steps

Read the displayed manifest for the executed sample counts. Missing quotes are not filled, and returns must join consecutive scheduled sessions. The fixed cohort and unverified adjustment convention limit investment inference. Classical PCA could reduce larger inputs; QPCA has a different data-access and readout model and is not required for this small case. The notebook is a teaching interface, not a prescribed software standard. To rebuild from another authorised archive, use the data-preparation module; use provider-managed authentication for any later hardware extension and keep literal tokens out of shared cells, scripts and outputs.